# Performance comparison between Swiftlet and Lark parsing library

Import required library

In [1]:
import random as rnd
import timeit
import swiftlet
import lark
import cProfile

Text expression generator

In [2]:
def gen_text_expr(n):
    expr = "{}".format(rnd.randint(1, n))
    opt = ["+", "-", "*", "/"]

    for _ in range(n):
        expr += " {} {}".format(rnd.choice(opt), str(rnd.randint(1, n)))
    return expr.strip()

texts = [gen_text_expr(10 * i) for i in range(1, 100)]

## Swiftlet Parser function

In [3]:
def parser_swiftlet(text: str):
    _grammar = """
    start: expr
    expr: expr "+" factors -> add
        | expr "-" factors -> sub
        | factors

    factors: factors "*" INT -> mul
        | factors "/" INT -> div
        | INT

    %import (WS, INT)
    %ignore WS
    """
    parser = swiftlet.Swiftlet(grammar=_grammar, algorithm="clr")
    ast = parser.parse(text)
    return ast

## Lark Parser function

In [4]:
def parser_lark(text: str):
    _grammar = """
    start: expr
    expr: expr "+" factors -> add
        | expr "-" factors -> sub
        | factors

    factors: factors "*" INT -> mul
        | factors "/" INT -> div
        | INT

    %import common.WS
    %import common.INT
    %ignore WS
    """
    _lark_parser = lark.Lark(_grammar, parser="lalr", keep_all_tokens=True)
    lark_ast = _lark_parser.parse(text=text)
    return lark_ast

In [5]:
number = 1000

In [6]:
for txt in texts[::20]:
    swiftlet_time = 1000 * timeit.timeit(lambda: parser_swiftlet(txt), number=number) / number
    lark_time = 1000 * timeit.timeit(lambda: parser_lark(txt), number=number) / number
    print('Length of text for parsing: ', len(txt),
          ', and time took to parse (swiftlet/lark): ({:.6f} ms / {:.6f} ms)'.format(swiftlet_time, lark_time),
          ', and percentage: {:.3f}%'.format(100 * (lark_time - swiftlet_time)/lark_time))

Length of text for parsing:  44 , and time took to parse (swiftlet/lark): (1.339289 ms / 6.615322 ms) , and percentage: 79.755%
Length of text for parsing:  1152 , and time took to parse (swiftlet/lark): (1.348233 ms / 5.603716 ms) , and percentage: 75.940%
Length of text for parsing:  2351 , and time took to parse (swiftlet/lark): (1.331995 ms / 5.972554 ms) , and percentage: 77.698%
Length of text for parsing:  3561 , and time took to parse (swiftlet/lark): (1.562863 ms / 7.268933 ms) , and percentage: 78.499%
Length of text for parsing:  4750 , and time took to parse (swiftlet/lark): (1.847089 ms / 8.255513 ms) , and percentage: 77.626%


In [7]:
# -------------- Swiftlet ------------------ #
grammar_swiftlet = """
    start: expr
    expr: expr "+" factors -> add
        | expr "-" factors -> sub
        | factors

    factors: factors "*" INT -> mul
        | factors "/" INT -> div
        | INT

    %import (WS, INT)
    %ignore WS
    """
swiftlet_parser = swiftlet.Swiftlet(grammar=grammar_swiftlet, algorithm="clr")

# -------------- Lark ------------------ #
grammar = """
    start: expr
    expr: expr "+" factors -> add
        | expr "-" factors -> sub
        | factors

    factors: factors "*" INT -> mul
        | factors "/" INT -> div
        | INT

    %import common.WS
    %import common.INT
    %ignore WS
    """
lark_parser = lark.Lark(grammar, parser="lalr", keep_all_tokens=True)


In [8]:
for txt in texts[::20]:
    swiftlet_time = 1000 * timeit.timeit(lambda: swiftlet_parser.parse(txt), number=number) / number
    lark_time = 1000 * timeit.timeit(lambda: lark_parser.parse(txt), number=number) / number

    print('Length of text for parsing: ', len(txt), ', and time took to parse (swiftlet/lark): ({:.6f} ms / {:.6f} ms)'.format(swiftlet_time, lark_time), ', and percentage: {:.3f}%'.format(100 * (lark_time - swiftlet_time)/lark_time))


Length of text for parsing:  44 , and time took to parse (swiftlet/lark): (0.013478 ms / 0.058982 ms) , and percentage: 77.149%
Length of text for parsing:  1152 , and time took to parse (swiftlet/lark): (0.249816 ms / 1.137858 ms) , and percentage: 78.045%
Length of text for parsing:  2351 , and time took to parse (swiftlet/lark): (0.475799 ms / 2.194200 ms) , and percentage: 78.316%
Length of text for parsing:  3561 , and time took to parse (swiftlet/lark): (0.728913 ms / 3.287339 ms) , and percentage: 77.827%
Length of text for parsing:  4750 , and time took to parse (swiftlet/lark): (0.979011 ms / 4.413845 ms) , and percentage: 77.820%


In [9]:
cProfile.run('parser_swiftlet(texts[98])')

         5 function calls in 0.003 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    0.002    0.002 2891851176.py:1(parser_swiftlet)
        1    0.000    0.000    0.003    0.003 <string>:1(<module>)
        1    0.000    0.000    0.003    0.003 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.001    0.001    0.001    0.001 {method 'parse' of 'swiftlet._core.Swiftlet' objects}




In [10]:
cProfile.run('parser_lark(texts[98])')

         106685 function calls (105231 primitive calls) in 0.032 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.032    0.032 595936109.py:1(parser_lark)
        2    0.000    0.000    0.000    0.000 <frozen _collections_abc>:576(_from_iterable)
        2    0.000    0.000    0.000    0.000 <frozen _collections_abc>:599(__or__)
       10    0.000    0.000    0.000    0.000 <frozen _collections_abc>:602(<genexpr>)
        8    0.000    0.000    0.000    0.000 <frozen abc>:117(__instancecheck__)
        1    0.000    0.000    0.001    0.001 <frozen importlib._bootstrap_external>:1127(get_data)
        1    0.000    0.000    0.000    0.000 <frozen importlib.util>:74(find_spec)
        1    0.000    0.000    0.000    0.000 <frozen posixpath>:150(dirname)
        4    0.000    0.000    0.000    0.000 <frozen posixpath>:41(_get_sep)
        3    0.000    0.000    0.000    0.000 <frozen posixpath>:

In [11]:
len(texts)

99

In [12]:
ast_lark = parser_lark("4 + 3 * 6 - 8 / 2")
print(ast_lark.pretty())

start
  sub
    add
      expr
        factors	4
      +
      mul
        factors	3
        *
        6
    -
    div
      factors	8
      /
      2



In [13]:
ast_rust = parser_swiftlet("4 + 3 * 6 - 8 / 2")
ast_rust.pretty_print()

start
   sub
     add
       expr
         factors
             4
       +
       mul
         factors
             3
         *
         6
     -
     div
       factors
           8
       /
       2
